In [ ]:
# Pre config 
from dotenv import load_dotenv
import os   
from crewai import Agent, Task, Crew, LLM
load_dotenv()
if os.environ.get("opneai_api", False):
    llm = "gpt-4o-mini"
elif os.environ.get("glm_key", False):
    llm = LLM(
    api_key=os.environ.get("glm_key"),
    base_url="https://api.z.ai/api/paas/v4/",
    model="glm-4.5",
    temperature=0.7
    )
else:
    llm = LLM(
        api_key='',
        base_url="http://localhost:11434",
        model="qwen3.5:0.8b",
        temperature=0.7
        )
    raise ValueError("No API key found for OpenAI or GLM. Please set the environment variable 'opneai_api' or 'glm_key'.")

def get_web_search_tool(url):
    # Create web RAG tool
    if os.environ.get("opneai_api", False):
        from crewai_tools import WebsiteSearchTool
        web_rag_tool = WebsiteSearchTool(website = url)
    else:
        import os
        if "OPENAI_API_KEY" in os.environ:
            del os.environ["OPENAI_API_KEY"]
        os.environ["OPENAI_API_KEY"] = "invalid"
        from chromadb.utils.embedding_functions.ollama_embedding_function import OllamaEmbeddingFunction
        import chromadb.utils.embedding_functions.openai_embedding_function as openai_ef_module
        class PatchedOllamaEmbeddingFunction(OllamaEmbeddingFunction):
            def __init__(self, *args, **kwargs):
                super().__init__(
                    url="http://localhost:11434",
                    model_name="nomic-embed-text"
                )

        openai_ef_module.OpenAIEmbeddingFunction = PatchedOllamaEmbeddingFunction
        from crewai_tools import WebsiteSearchTool
    
        web_rag_tool = WebsiteSearchTool(
            website=url,
            llm=llm,
            collection_name="ollama_collection"
        )
    return web_rag_tool

c:\Users\wasie\anaconda3\Lib\site-packages\paramiko\pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
c:\Users\wasie\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.Blowfish and will be removed from this module in 45.0.0.
  "class": algorithms.Blowfish,
c:\Users\wasie\anaconda3\Lib\site-packages\paramiko\transport.py:243: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "class": algorithms.TripleDES,


In [ ]:
# =============================================================================
# Session 05 3: L1 - SINGLE AGENT WITH CREWAI
# =============================================================================

from crewai import Agent

print("\n" + "="*60)
print("SINGLE AGENT DEMONSTRATION")
print("="*60)

topic="Multi-agent Systems in Artificial Intelligence"
general_agent = Agent(
    role="Content blogger",
    goal=f"Plan a blog content on {topic}",
    backstory=f"You're working on a blog article about the topic: {topic}. "
                "You collect information that helps the audience learn something "
                "and make informed decisions. Your work is the basis for "
                "the Content Writer to write an article on this topic.",
    allow_delegation=False,
    llm=llm,
    verbose=True
)

result = await general_agent.kickoff(topic)
print(result)


SINGLE AGENT DEMONSTRATION
┌─────────────────────────── 🤖 LiteAgent Started ────────────────────────────┐
│                                                                             │
│  LiteAgent Started                                                          │
│  Role: Content blogger                                                      │
│  Status: In Progress                                                        │
│                                                                             │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘

┌─────────────────────────── 🤖 LiteAgent Started ────────────────────────────┐
│                                                                             │
│  LiteAgent Session Started                                                  │
│  Name: Content blogger                                                      │
│  id: 462c95

In [ ]:
print(result)

# Blog Content Plan: Multi-agent Systems in Artificial Intelligence

## Article Outline

### 1. Introduction
- **Hook**: Start with a compelling example (e.g., coordinated drones, smart traffic systems, or game AI)
- **Definition**: What are multi-agent systems (MAS) in AI?
- **Importance**: Why MAS matter in modern AI development
- **Article overview**: What readers will learn

### 2. Understanding Multi-Agent Systems
- **Core concepts**:
  - What constitutes an "agent" in AI
  - Key characteristics of MAS
  - Autonomy, local views, and decentralization
  - How MAS differ from single-agent systems
- **Historical development** of MAS
- **Theoretical foundations**

### 3. Key Components and Architecture
- **Agent architecture types**:
  - Reactive agents
  - Deliberative agents
  - Hybrid agents
- **Communication protocols**:
  - Direct vs. indirect communication
  - Communication languages (KQML, FIPA ACL)
- **Coordination mechanisms**:
  - Cooperation, competition, negotiation
  - Auc

In [ ]:
# =============================================================================
# SECTION 4: L3 - SINGLE AGENT WITH TOOLS
# =============================================================================
from crewai import Agent, Task, Crew
from crewai_tools import WebsiteSearchTool

print("\n" + "="*60)
print("L3: SINGLE AGENT WITH TOOLS DEMONSTRATION")
print("="*60)

# Create web RAG tool
web_rag_tool = get_web_search_tool("https://www.deeplearning.ai/the-batch/")

# Create researcher agent with tool
researcher = Agent(
    role='Latest Trends Analyst',
    goal='Provide up-to-date trending developments in AI',
    backstory='An expert analyst with a keen eye for market trends.',
    tools=[web_rag_tool],
    llm=llm,
    verbose=True
)

# Define task
research = Task(
    description='Research the latest trends in the AI industry.',
    expected_output='Provide the top 3 trending in the AI industry.',
    agent=researcher
)

# Create crew with planning enabled
crew = Crew(
    agents=[researcher],
    tasks=[research],
    verbose=True,
    planning=True,
)

result = await crew.kickoff()
print(result)

In [ ]:
# =============================================================================
# SECTION 5: L4 - MULTI-AGENT RESEARCH AND WRITING
# =============================================================================
from crewai import Agent, Task, Crew

print("\n" + "="*60)
print("L4: MULTI-AGENT RESEARCH AND WRITING DEMONSTRATION")
print("="*60)

topic="Artificial Intelligence"
# llm = "gpt-4o-mini"

# Create agents
planner = Agent(
    role="Content Planner",
    goal=f"Plan engaging and factually accurate content on {topic}",
    backstory=f"You're working on planning a blog article about the topic: {topic}. "
                "You collect information that helps the audience learn something "
                "and make informed decisions. Your work is the basis for "
                "the Content Writer to write an article on this topic.",
    allow_delegation=False,
    llm=llm,
    verbose=True
)

writer = Agent(
    role="Content Writer",
    goal=f"Write insightful and factually accurate opinion piece about {topic}",
    backstory=f"You're working on writing a new opinion piece about the topic: {topic}. "
                "You base your writing on the work of the Content Planner.",
    allow_delegation=False,
    llm=llm,
    verbose=True
)

editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with the writing style of the organization.",
    backstory="You are an editor who receives a blog post from the Content Writer. "
                "Your goal is to review the blog post to ensure that it follows journalistic best practices.",
    allow_delegation=False,
    llm=llm,
    verbose=True
)

# Create tasks
plan = Task(
    description=(
        f"1. Prioritize the latest trends, key players, and noteworthy news on {topic}.\n"
        f"2. Identify the target audience, considering their interests and pain points.\n"
        f"3. Develop a detailed content outline including an introduction, key points, and a call to action.\n"
        f"4. Include SEO keywords and relevant data or sources."
    ),
    expected_output=f"A comprehensive content plan document with an outline, audience analysis, SEO keywords, and resources.",
    agent=planner,
)

write = Task(
    description=(
        f"1. Use the content plan to craft a compelling blog post on {topic}.\n"
        f"2. Incorporate SEO keywords naturally.\n"
        f"3. Sections/Subtitles are properly named in an engaging manner.\n"
        f"4. Ensure the post is structured with an engaging introduction, insightful body, and a summarizing conclusion.\n"
        f"5. Proofread for grammatical errors and alignment with the brand's voice."
    ),
    expected_output="A well-written blog post in markdown format, ready for publication.",
    agent=writer,
)

edit = Task(
    description="Proofread the given blog post for grammatical errors and alignment with the brand's voice.",
    expected_output="A well-written blog post in markdown format, ready for publication.",
    agent=editor
)

# Create and run crew
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    verbose=1
)

result = crew.kickoff(inputs={"topic": topic})
print(result)

In [ ]:
# =============================================================================
# SECTION 6: L5 - AGENT DELEGATION AND MEMORY
# =============================================================================
from crewai import Agent, Task, Crew
from crewai_tools import ScrapeWebsiteTool

print("\n" + "="*60)
print("L5: AGENT DELEGATION AND MEMORY DEMONSTRATION")
print("="*60)

customer="ResolveAI"
person="Usman" 
inquiry="I need help with setting up a CrewAI for safe city"
# llm = "gpt-4o-mini"

# Create documentation scrape tool
docs_scrape_tool = ScrapeWebsiteTool(
    website_url="https://docs.crewai.com/concepts/crews"
)

# Create agents
support_agent = Agent(
    role="Senior Support Representative",
    goal="Be the most friendly and helpful support representative in your team",
    backstory=f"You work at crewAI and are now working on providing support to {customer}, "
                f"a super important customer for your company. "
                "You need to make sure that you provide the best support! "
                "Make sure to provide full complete answers, and make no assumptions.",
    allow_delegation=False,
    llm=llm,
    verbose=True
)

support_quality_assurance_agent = Agent(
    role="Support Quality Assurance Specialist",
    goal="Get recognition for providing the best support quality assurance in your team",
    backstory=f"You work at crewAI and are now working with your team on a request from {customer} "
                "ensuring that the support representative is providing the best support possible.",
    verbose=True,
    allow_delegation=True,
    llm=llm,
)

# Create tasks
inquiry_resolution = Task(
    description=f"{customer} just reached out with a super important ask:\n"
                f"{inquiry}\n\n"
                f"{person} from {customer} is the one that reached out. "
                "Make sure to use everything you know to provide the best support possible.",
    expected_output="A detailed, informative response to the customer's inquiry.",
    tools=[docs_scrape_tool],
    agent=support_agent,
)

quality_assurance_review = Task(
    description=f"Review the response drafted by the Senior Support Representative for {customer}'s inquiry. "
                "Ensure that the answer is comprehensive, accurate, and adheres to the high-quality standards.",
    expected_output="A final, detailed, and informative response ready to be sent to the customer.",
    agent=support_quality_assurance_agent,
)

# Create crew with memory
crew = Crew(
    agents=[support_agent, support_quality_assurance_agent],
    tasks=[inquiry_resolution, quality_assurance_review],
    verbose=1,
    memory=True
)

result = crew.kickoff()
print(result)